# 03 · Queries de verificación — Smart Solutions Dietta

**Team Challenge SQL · Parte II** — Alejandro Dietta

Queries analíticas sobre los datos cargados en `02_generate_data.ipynb` para comprobar que el modelo responde a las necesidades de negocio de la empresa:

| # | Query | Necesidad de negocio |
|---|---|---|
| 1 | Ingresos por mes | Identificar tendencias temporales de ventas |
| 2 | Top 10 productos más vendidos | Conocer qué productos compran los clientes |
| 3 | Ingresos, coste y margen por categoría | Calcular márgenes y rentabilidad por categoría |
| 4 | Clientes e ingresos por país y canal | Segmentar clientes por país y canal de adquisición |
| 5 | Tiempo medio de entrega por país | Controlar los envíos |
| 6 | Valoración media por categoría | Medir la satisfacción de los clientes |

**Criterios comunes:**
- **Ingreso** de una línea = `quantity × unit_price × (1 − discount_pct)`, usando el precio en el momento de la compra.
- Los pedidos `cancelled` y `returned` **no cuentan como ingreso**.

## Configuración

Misma conexión que en los notebooks anteriores. `DATASET` guarda `proyecto.dataset` para escribir los nombres completos de las tablas en las queries.

In [38]:
import os
import warnings
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from google.cloud import bigquery
from google.oauth2 import service_account

# google-cloud-bigquery muestra un aviso informativo que no afecta a las queries:
# - BigQuery Storage: sin ese módulo, los resultados se descargan por la API REST normal.
# No se instala esa librería porque, al hacerlo, aparecen nuevos avisos de obsolescencia que
# piden sustituir client.query().to_dataframe() por pandas_gbq.read_gbq(). Se mantiene el código
# de la guía del bootcamp y las dependencias de requirements.txt, y se silencia solo este aviso.
warnings.filterwarnings("ignore", message=".*BigQuery Storage module not found.*")

env_path = find_dotenv(usecwd=True)
load_dotenv(env_path)
ROOT = Path(env_path).parent

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")
CREDENTIALS_PATH = ROOT / os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

credentials = service_account.Credentials.from_service_account_file(CREDENTIALS_PATH)
client = bigquery.Client(project=PROJECT_ID, credentials=credentials)

DATASET = f"{PROJECT_ID}.{DATASET_ID}"

## 1. Ingresos por mes

¿Cómo evolucionan las ventas a lo largo del tiempo? Se agrupan los pedidos por mes (`FORMAT_TIMESTAMP('%Y-%m', ...)`) y se suman los ingresos de sus líneas.

In [39]:
query = f"""
SELECT FORMAT_TIMESTAMP('%Y-%m', customer_order.order_date) AS month,
    COUNT(DISTINCT customer_order.order_id) AS total_orders,
    ROUND(SUM(item.quantity * item.unit_price * (1 - item.discount_pct)), 2) AS revenue
FROM `{DATASET}.orders` AS customer_order
JOIN `{DATASET}.order_items` AS item ON item.order_id = customer_order.order_id
WHERE customer_order.status NOT IN ('cancelled', 'returned')
GROUP BY month
ORDER BY month;
"""

client.query(query).to_dataframe()

,month,total_orders,revenue
0,2024-01,4,5192.96
1,2024-02,2,3028.11
2,2024-03,12,15392.81
3,2024-04,15,15476.69
4,2024-05,20,32390.38
5,2024-06,28,40421.90
6,2024-07,46,53329.26
7,2024-08,38,55516.51
8,2024-09,35,49580.58
9,2024-10,52,59586.79


**Resultado:** las ventas crecen de forma continua, hasta ~200.000 €/mes a finales de 2025.

## 2. Top 10 productos más vendidos

¿Qué productos compran más los clientes? Se ordenan por unidades vendidas y se muestran también sus ingresos y su categoría.

In [40]:
query = f"""
SELECT product.name AS product_name, category.name AS category_name,
    SUM(item.quantity) AS units_sold,
    ROUND(SUM(item.quantity * item.unit_price * (1 - item.discount_pct)), 2) AS revenue
FROM `{DATASET}.order_items` AS item
JOIN `{DATASET}.orders` AS customer_order ON customer_order.order_id = item.order_id
JOIN `{DATASET}.products` AS product ON product.product_id = item.product_id
JOIN `{DATASET}.categories` AS category ON category.category_id = product.category_id
WHERE customer_order.status NOT IN ('cancelled', 'returned')
GROUP BY product.name, category.name
ORDER BY units_sold DESC
LIMIT 10;
"""

client.query(query).to_dataframe()

,product_name,category_name,units_sold,revenue
0,Micrófono USB VoiceCast,Periféricos,94,47494.88
1,Smartphone Nova X5,Smartphones,94,82900.16
2,Smartwatch Outdoor Trek,Wearables,89,6536.04
3,Smartphone Pixo 8a,Smartphones,88,107423.84
4,Smartphone Pixo 8,Smartphones,87,31829.93
5,Cargador inalámbrico MagPad,Accesorios,85,9583.84
6,Monitor 24 ViewFull,Periféricos,85,43125.46
7,Smartwatch Pulse 4,Wearables,85,22201.82
8,Volante RaceWheel,Gaming,85,27327.12
9,Tablet Kids Tab 8,Tablets,83,45513.03


**Resultado:** las unidades vendidas están muy repartidas entre productos (83–94 unidades en el top 10), pero los ingresos no: con un volumen similar, un smartphone como el *Pixo 8a* genera más de 100.000 € y un smartwatch como el *Outdoor Trek* unos 6.500 €. Vender más unidades no implica facturar más.

## 3. Ingresos, coste y margen por categoría

¿Qué categorías son más rentables? El **coste** de cada línea es `quantity × products.cost` (el modelo guarda el coste actual del producto, no el histórico) y el **margen** es la diferencia entre ingresos y coste.

Se usa un `WITH` (*CTE*) para calcular primero los totales por categoría y después, a partir de ellos, el margen y el porcentaje de margen, sin repetir las fórmulas.

In [41]:
query = f"""
WITH category_sales AS (
    SELECT category.name AS category_name,
        SUM(item.quantity * item.unit_price * (1 - item.discount_pct)) AS revenue,
        SUM(item.quantity * product.cost) AS cost
    FROM `{DATASET}.order_items` AS item
    JOIN `{DATASET}.orders` AS customer_order ON customer_order.order_id = item.order_id
    JOIN `{DATASET}.products` AS product ON product.product_id = item.product_id
    JOIN `{DATASET}.categories` AS category ON category.category_id = product.category_id
    WHERE customer_order.status NOT IN ('cancelled', 'returned')
    GROUP BY category.name
)
SELECT category_name, ROUND(revenue, 2) AS revenue, ROUND(cost, 2) AS cost,
    ROUND(revenue - cost, 2) AS margin,
    ROUND((revenue - cost) / revenue * 100, 1) AS margin_pct
FROM category_sales
ORDER BY margin DESC;
"""

client.query(query).to_dataframe()

,category_name,revenue,cost,margin,margin_pct
0,Laptops,790226.83,571441.03,218785.80,27.7
1,Smartphones,511723.88,327369.21,184354.67,36.0
2,Tablets,345613.48,237362.01,108251.47,31.3
3,Gaming,238956.19,158743.74,80212.45,33.6
4,Periféricos,177877.88,122907.27,54970.61,30.9
5,Wearables,121934.73,80119.15,41815.58,34.3
6,Audio,133322.40,96252.05,37070.35,27.8
7,Accesorios,51433.46,36689.52,14743.94,28.7


**Resultado:** **Laptops** es la categoría con más ingresos y más margen en euros, pero con uno de los porcentajes de margen más bajos (27,7 %). **Smartphones** tiene el mejor porcentaje (36 %). **Accesorios** es la categoría que menos aporta por su precio bajo.

## 4. Clientes e ingresos por país y canal de adquisición

¿De dónde vienen los clientes y cuáles generan más ingresos?

Se parte de `customers` con `LEFT JOIN` para contar también a los clientes que aún no han comprado. La condición de excluir pedidos cancelados y devueltos va en el `ON` del `JOIN` y no en el `WHERE`: en el `WHERE` eliminaría a los clientes sin pedidos válidos, y el `LEFT JOIN` dejaría de tener efecto.

In [42]:
query = f"""
SELECT customer.country, customer.acquisition_channel,
    COUNT(DISTINCT customer.customer_id) AS customers,
    ROUND(SUM(item.quantity * item.unit_price * (1 - item.discount_pct)), 2) AS revenue
FROM `{DATASET}.customers` AS customer
LEFT JOIN `{DATASET}.orders` AS customer_order ON customer_order.customer_id = customer.customer_id
    AND customer_order.status NOT IN ('cancelled', 'returned')
LEFT JOIN `{DATASET}.order_items` AS item ON item.order_id = customer_order.order_id
GROUP BY customer.country, customer.acquisition_channel
ORDER BY customer.country, revenue DESC;
"""

client.query(query).to_dataframe()

,country,acquisition_channel,customers,revenue
0,Alemania,organic,32,169490.11
1,Alemania,paid_ads,24,122079.71
2,Alemania,referral,13,70992.55
3,Alemania,social_media,17,68796.34
4,Alemania,email,13,48887.14
5,España,organic,64,309245.98
6,España,paid_ads,45,206558.74
7,España,social_media,36,161920.27
8,España,email,22,129317.31
9,España,referral,19,77569.96


**Resultado:** **España** es el principal mercado en clientes e ingresos. El canal **organic** es el que más ingresos aporta en todos los países y también el que más clientes capta, salvo en Francia, donde lo supera **paid_ads**. **Referral** y **email** suelen ser los canales más pequeños.

## 5. Tiempo medio de entrega por país

¿Cuánto tardan en llegar los pedidos? Se calcula con `DATE_DIFF` los días entre la fecha del pedido y la de entrega, solo para los pedidos que tienen fecha de entrega (entregados y devueltos).

In [43]:
query = f"""
SELECT shipping_country,
    COUNT(*) AS delivered_orders,
    ROUND(AVG(DATE_DIFF(delivered_date, DATE(order_date), DAY)), 1) AS avg_delivery_days
FROM `{DATASET}.orders`
WHERE delivered_date IS NOT NULL
GROUP BY shipping_country
ORDER BY avg_delivery_days;
"""

client.query(query).to_dataframe()

,shipping_country,delivered_orders,avg_delivery_days
0,Francia,306,5.8
1,Alemania,302,6.0
2,España,528,6.1
3,Portugal,183,6.4
4,Italia,152,6.4


**Resultado:** el tiempo medio de entrega es de unos **6 días** en todos los países (5,8–6,4). Las diferencias son pequeñas porque los plazos de envío y entrega se generaron con la misma distribución para todos los países.

## 6. Valoración media por categoría

¿Qué categorías valoran mejor los clientes? Las valoraciones se relacionan con las líneas de pedido, así que se llega a la categoría a través de `order_items` → `products` → `categories`.

In [44]:
query = f"""
SELECT category.name AS category_name,
    COUNT(*) AS total_reviews,
    ROUND(AVG(review.rating), 2) AS avg_rating
FROM `{DATASET}.reviews` AS review
JOIN `{DATASET}.order_items` AS item ON item.order_item_id = review.order_item_id
JOIN `{DATASET}.products` AS product ON product.product_id = item.product_id
JOIN `{DATASET}.categories` AS category ON category.category_id = product.category_id
GROUP BY category.name
ORDER BY avg_rating DESC;
"""

client.query(query).to_dataframe()

,category_name,total_reviews,avg_rating
0,Smartphones,156,3.99
1,Accesorios,149,3.97
2,Laptops,151,3.95
3,Wearables,120,3.95
4,Gaming,129,3.94
5,Audio,148,3.92
6,Periféricos,148,3.86
7,Tablets,114,3.84


**Resultado:** todas las categorías tienen una valoración media cercana a **4 sobre 5** (3,84–3,99), con **Smartphones** en cabeza y **Tablets** en último lugar. Las notas son parecidas porque se generaron con la misma distribución para todas las categorías.